#### EDA 06: Products table and category translation
**Files:** `olist_products_dataset.csv` and `product_category_name_translation.csv`  |  **Goal:** understand the product attributes and categories, using these two files on their own. No joins happen here. Cross-table checks are listed at the end.

**Flow:** Grain → Duplicates → Missing → Listing fields → Weight and size → Categories → Category vs attributes → Outliers → Translation table → Summary

#### Load and first look
**Question:** how big is the table and what does a row look like?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import unicodedata

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RAW = Path("../../data/raw")                      # path from the notebook folder to the raw data

products_raw = pd.read_csv(RAW / "olist_products_dataset.csv")
products = products_raw.copy()                    # all changes happen on the copy, raw stays untouched

print("shape:", products.shape)
print(products.dtypes)
products.head(3)

shape: (32951, 9)
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


- 32,951 rows and 9 columns: product id, category, four listing fields (name length, description length, photos) and weight plus three dimensions.
- The numeric columns load as decimals (float) because they contain missing values.
- Two column names are misspelled in the source (`product_name_lenght`, `product_description_lenght`).
- **Decision:** rename them to `..._length` in the working copy. The raw file is not touched.

In [2]:
products = products.rename(columns={"product_name_lenght": "product_name_length",
                                    "product_description_lenght": "product_description_length"})
list(products.columns)

['product_id',
 'product_category_name',
 'product_name_length',
 'product_description_length',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

#### Keys and duplicates
**Question:** is `product_id` a valid key, and do different products share the same attributes?

In [3]:
print("product_id unique         :", products["product_id"].is_unique)
print("fully duplicated rows     :", products.duplicated().sum())

# Rows identical to another product in every attribute, only the id differs
attrs = products.drop(columns="product_id")
extra_copies = attrs.duplicated().sum()
same_attrs = attrs.duplicated(keep=False)
print("extra copies (same attributes, different product_id):", extra_copies)
print("rows sharing all attributes with another product    :", same_attrs.sum(),
      "| of these with category missing:", products.loc[same_attrs, "product_category_name"].isna().sum())

product_id unique         : True
fully duplicated rows     : 0
extra copies (same attributes, different product_id): 695
rows sharing all attributes with another product    : 1158 | of these with category missing: 133


- `product_id` is unique and no row is fully duplicated. One row is one product.
- 1,158 rows share all attributes with another product, which makes 695 extra copies. In 133 of those rows the category is missing, so part of this is the empty block (nothing left to tell products apart).
- Identical weight, size and text lengths do not prove the same product. Different products can share such values.
- **Decision:** keep all products. Merging them would be a guess.

#### Missing values
**Question:** which columns have gaps, and do they go missing together?

In [4]:
# Column-wise missing count and percentage, for every column (zeros included)
missing = pd.DataFrame({"n_rows": len(products),
                        "n_missing": products.isna().sum(),
                        "pct_missing": (products.isna().mean() * 100).round(2)})
display(missing.sort_values("n_missing", ascending=False))

LISTING = ["product_category_name", "product_name_length", "product_description_length", "product_photos_qty"]
PHYSICAL = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

no_listing = products[LISTING].isna().all(axis=1)           # all four listing fields empty
no_physical = products[PHYSICAL].isna().all(axis=1)         # weight and all dimensions empty
partly_listing = products[LISTING].isna().any(axis=1) & ~no_listing
partly_physical = products[PHYSICAL].isna().any(axis=1) & ~no_physical
print("products with only SOME listing fields missing :", partly_listing.sum())
print("products with only SOME physical fields missing:", partly_physical.sum())

display(pd.crosstab(no_listing, no_physical).rename(index={False: "listing present", True: "listing missing"},
                                                    columns={False: "weight/size present", True: "weight/size missing"}))

# Flag for later (the listing block is a collection gap, not an event that never happened)
products["no_listing_info"] = no_listing

,n_rows,n_missing,pct_missing
product_category_name,32951,610,1.85
product_description_length,32951,610,1.85
product_name_length,32951,610,1.85
product_photos_qty,32951,610,1.85
product_weight_g,32951,2,0.01
product_height_cm,32951,2,0.01
product_length_cm,32951,2,0.01
product_width_cm,32951,2,0.01
product_id,32951,0,0.00


products with only SOME listing fields missing : 0
products with only SOME physical fields missing: 0


col_0,weight/size present,weight/size missing
row_0,,
listing present,32340,1
listing missing,609,1


- Missing values come in two blocks, and each block is all-or-nothing (no product has only some of its fields missing).
- Block 1: 610 products (1.85%) have category, name length, description length and photos all missing. The listing information was not filled in.
- Block 2: 2 products have weight and all three dimensions missing. One of them is also in block 1.
- This is a collection gap (information that exists for a real product but was not recorded), not an event that never happened.
- **Decision:** keep all rows and set `no_listing_info`. Do not fill the category or the lengths. A model can use the flag, and an unknown category stays "unknown". The 2 products without weight and size stay empty.
- Whether these products relate to worse outcomes is checked after the joins.

#### Listing fields: name, description and photos
**Question:** what do the listing fields look like?

In [5]:
display(products[["product_name_length", "product_description_length", "product_photos_qty"]]
        .describe(percentiles=[0.01, 0.5, 0.95, 0.99]).T.round(1))

photos = products["product_photos_qty"].dropna()
photo_dist = photos.clip(upper=9).value_counts().sort_index().rename("n_products").to_frame()
photo_dist.index = [str(int(i)) if i < 9 else "9+" for i in photo_dist.index]
photo_dist.index.name = "photos"
photo_dist["pct"] = (photo_dist["n_products"] / len(photos) * 100).round(1)
display(photo_dist)
print("products with more than 10 photos:", (photos > 10).sum(), "| maximum:", photos.max())
print("products with 0 photos           :", (photos == 0).sum())

,count,mean,std,min,1%,50%,95%,99%,max
product_name_length,32341.0,48.5,10.2,5.0,20.0,51.0,60.0,63.0,76.0
product_description_length,32341.0,771.5,635.1,4.0,84.0,595.0,2063.0,3286.2,3992.0
product_photos_qty,32341.0,2.2,1.7,1.0,1.0,1.0,6.0,8.0,20.0


,n_products,pct
photos,,
1,16489,51.0
2,6263,19.4
3,3860,11.9
4,2428,7.5
5,1484,4.6
6,968,3.0
7,343,1.1
8,192,0.6
9+,314,1.0


products with more than 10 photos: 114 | maximum: 20.0
products with 0 photos           : 0


- Name length: median 51 characters, 99th percentile 63, maximum 76.
- Description length: median 595, mean 771.5, 99th percentile 3,286, maximum 3,992 (close to 4,000). It is right-skewed.
- Photos: the median is 1 and the maximum 20. About half of the products (16,489 of 32,341, 51.0%) have one photo only. 114 products have more than 10.
- Every product with listing information has at least one photo (no zero) and at least a few characters of description.

#### Weight and size
**Question:** what do weight and dimensions look like, and are there impossible values?

In [6]:
products["volume_cm3"] = (products["product_length_cm"] * products["product_height_cm"] * products["product_width_cm"])   # box volume

display(products[PHYSICAL + ["volume_cm3"]].describe(percentiles=[0.01, 0.5, 0.95, 0.99]).T.round(1))

print("products with weight == 0 :", (products["product_weight_g"] == 0).sum())
print("products above 10 kg      :", (products["product_weight_g"] > 10000).sum())
print("products above 30 kg      :", (products["product_weight_g"] > 30000).sum())

print("\nweight == 0")
display(products.loc[products["product_weight_g"] == 0, ["product_category_name"] + PHYSICAL])
print("heaviest products")
products.nlargest(3, "product_weight_g")[["product_category_name"] + PHYSICAL]

,count,mean,std,min,1%,50%,95%,99%,max
product_weight_g,32949.0,2276.5,4282.0,0.0,60.0,700.0,10850.0,22538.0,40425.0
product_length_cm,32949.0,30.8,16.9,7.0,16.0,25.0,65.0,100.0,105.0
product_height_cm,32949.0,16.9,13.6,2.0,2.0,13.0,44.0,69.0,105.0
product_width_cm,32949.0,23.2,12.1,6.0,11.0,20.0,47.0,63.0,118.0
volume_cm3,32949.0,16564.1,27057.0,168.0,352.0,6840.0,63369.0,135917.4,296208.0


products with weight == 0 : 4
products above 10 kg      : 1891
products above 30 kg      : 1

weight == 0


,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,cama_mesa_banho,0.0,30.0,25.0,30.0
13683,cama_mesa_banho,0.0,30.0,25.0,30.0
14997,cama_mesa_banho,0.0,30.0,25.0,30.0
32079,cama_mesa_banho,0.0,30.0,25.0,30.0


heaviest products


,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
25166,cama_mesa_banho,40425.0,13.0,65.0,28.0
344,moveis_escritorio,30000.0,50.0,50.0,30.0
509,utilidades_domesticas,30000.0,76.0,51.0,51.0


- Weight: median 700 g, mean 2,276.5 g, 95th percentile 10,850 g, 99th percentile 22,538 g, maximum 40,425 g. 1,891 products are above 10 kg and 1 is above 30 kg.
- Dimensions: length 7 to 105 cm, height 2 to 105 cm, width 6 to 118 cm. Box volume: median 6,840 cm3, 99th percentile 135,917 cm3, maximum 296,208 cm3.
- 4 products have a weight of 0 g, which is impossible. All four are `cama_mesa_banho` (bed, bath and table) with the same size (30 x 25 x 30 cm), so they look like one entry pattern.
- The heaviest product (40,425 g) is also `cama_mesa_banho` with a small box (13 x 65 x 28 cm). That is suspicious for bedding, but the other two heaviest are 30,000 g furniture and household items, which are plausible.
- **Decision:** keep all products. Treat weight 0 as unknown in the pipeline. Cap weight and size only inside the model pipeline.

#### Product categories
**Question:** how many categories are there, how concentrated are they, and is the text clean?

In [7]:
cat = products["product_category_name"].dropna()
cat_counts = cat.value_counts()

print("distinct categories           :", len(cat_counts))
print("products with a category      :", len(cat))
print("top 10 categories, share      :", f"{cat_counts.head(10).sum() / len(cat) * 100:.1f}%")
print("categories with 20 or fewer products:", (cat_counts <= 20).sum())
print("text check (upper case, spaces, non-ASCII):",
      cat.map(lambda s: s != s.lower() or s != s.strip() or not s.isascii()).sum())

top = pd.DataFrame({"n_products": cat_counts.head(10), "pct": (cat_counts.head(10) / len(cat) * 100).round(1)})
display(top)
print("5 smallest categories")
cat_counts.tail(5).rename("n_products").to_frame()

distinct categories           : 73
products with a category      : 32341
top 10 categories, share      : 64.1%
categories with 20 or fewer products: 13
text check (upper case, spaces, non-ASCII): 0


,n_products,pct
product_category_name,,
cama_mesa_banho,3029,9.4
esporte_lazer,2867,8.9
moveis_decoracao,2657,8.2
beleza_saude,2444,7.6
utilidades_domesticas,2335,7.2
automotivo,1900,5.9
informatica_acessorios,1639,5.1
brinquedos,1411,4.4
relogios_presentes,1329,4.1


5 smallest categories


,n_products
product_category_name,
fashion_roupa_infanto_juvenil,5
casa_conforto_2,5
pc_gamer,3
seguros_e_servicos,2
cds_dvds_musicais,1


- 73 distinct categories for 32,341 products that have one.
- The top 10 categories hold 64.1% of products. The biggest is `cama_mesa_banho` (bed, bath and table, 3,029 products), then `esporte_lazer` (sports and leisure, 2,867) and `moveis_decoracao` (furniture and decor, 2,657).
- The tail is long: 13 categories have 20 or fewer products. The smallest are `cds_dvds_musicais` (1 product), `seguros_e_servicos` (2) and `pc_gamer` (3).
- Names are clean: all lower case, no spaces, ASCII only. They are Portuguese and are translated in the translation table below.

#### Category vs attributes
**Question:** do the attributes differ by category, and are the numeric fields related?

In [8]:
by_cat = (products.groupby("product_category_name")
          .agg(n_products=("product_id", "size"),
               median_weight_g=("product_weight_g", "median"),
               median_volume_cm3=("volume_cm3", "median"),
               median_photos=("product_photos_qty", "median"),
               median_description=("product_description_length", "median"))
          .round(0))

print("8 largest categories")
display(by_cat.sort_values("n_products", ascending=False).head(8))

big = by_cat[by_cat["n_products"] >= 100].sort_values("median_weight_g")
print("lightest and heaviest categories (100+ products), by median weight")
display(pd.concat([big.head(3), big.tail(3)])[["n_products", "median_weight_g", "median_volume_cm3"]])

print("correlation between numeric fields")
products[["product_weight_g", "volume_cm3", "product_photos_qty", "product_description_length", "product_name_length"]].corr().round(2)

8 largest categories


,n_products,median_weight_g,median_volume_cm3,median_photos,median_description
product_category_name,,,,,
cama_mesa_banho,3029,1250.0,12000.0,1.0,365.0
esporte_lazer,2867,750.0,8000.0,1.0,733.0
moveis_decoracao,2657,1200.0,11520.0,2.0,582.0
beleza_saude,2444,523.0,4608.0,1.0,913.0
utilidades_domesticas,2335,1050.0,11880.0,1.0,543.0
automotivo,1900,900.0,7920.0,2.0,690.0
informatica_acessorios,1639,400.0,4056.0,1.0,578.0
brinquedos,1411,700.0,12075.0,2.0,587.0


lightest and heaviest categories (100+ products), by median weight


,n_products,median_weight_g,median_volume_cm3
product_category_name,,,
fashion_bolsas_e_acessorios,849,200.0,1584.0
telefonia,1134,200.0,1080.0
eletronicos,517,250.0,3420.0
climatizacao,124,3371.0,19450.0
moveis_sala,156,6150.0,33494.0
moveis_escritorio,309,11600.0,70200.0


correlation between numeric fields


,product_weight_g,volume_cm3,product_photos_qty,product_description_length,product_name_length
product_weight_g,1.00,0.80,0.04,0.06,0.04
volume_cm3,0.80,1.00,0.03,0.03,0.03
product_photos_qty,0.04,0.03,1.00,0.11,0.13
product_description_length,0.06,0.03,0.11,1.00,0.10
product_name_length,0.04,0.03,0.13,0.10,1.00


- Weight depends strongly on the category. Among categories with 100+ products, the lightest median weights are 200 g (`fashion_bolsas_e_acessorios` and `telefonia`) and 250 g (`eletronicos`). The heaviest are `moveis_escritorio` (office furniture, 11,600 g), `moveis_sala` (6,150 g) and `climatizacao` (3,371 g).
- Weight and box volume move together (correlation 0.80), as expected for physical objects.
- The listing fields hardly relate to weight or size (correlations of 0.03 to 0.06) and only slightly to each other (0.10 to 0.13).
- Category is therefore the best compact description of what a product is, and it carries the size information that freight depends on.

#### Outliers
**Question:** how many products are extreme in weight, and are they real?

In [9]:
w = products["product_weight_g"].dropna()
q1, q3 = w.quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)                              # IQR rule, upper fence
print(f"IQR upper fence for weight: {upper:,.0f} g")
print("products above the fence   :", (w > upper).sum(), f"({(w > upper).mean() * 100:.1f}%)")

by_cat_heavy = products.loc[products["product_weight_g"] > upper, "product_category_name"].value_counts().head(5)
by_cat_heavy.rename("n_heavy_products").to_frame()

IQR upper fence for weight: 4,300 g


products above the fence   : 4551 (13.8%)


,n_heavy_products
product_category_name,
moveis_decoracao,483
utilidades_domesticas,470
cama_mesa_banho,378
automotivo,372
esporte_lazer,302


- The IQR rule flags 4,551 products (13.8%) above 4,300 g. That share is too large for errors.
- The heavy products sit mainly in furniture and decor (483), household (470), bed and bath (378), auto (372) and sports (302), where heavy items are normal.
- **Decision:** keep all. Weight is naturally right-skewed, so cap or log it inside the model pipeline only.

#### Category translation table
**Question:** what does the translation file look like on its own? It is inspected separately here. How it matches the products table is checked after the joins.

In [10]:
cat_en_raw = pd.read_csv(RAW / "product_category_name_translation.csv")
cat_en = cat_en_raw.copy()

print("shape:", cat_en.shape)
print("missing values              :", cat_en.isna().sum().sum())
print("fully duplicated rows       :", cat_en.duplicated().sum())
print("Portuguese name unique      :", cat_en["product_category_name"].is_unique)
print("English name unique         :", cat_en["product_category_name_english"].is_unique)
print("English names with '_'      :", cat_en["product_category_name_english"].str.contains("_").sum())

same = cat_en[cat_en["product_category_name"] == cat_en["product_category_name_english"]]
print("\nnames identical in both languages:", len(same))
display(same)
cat_en.head(5)

shape: (71, 2)
missing values              : 0
fully duplicated rows       : 0
Portuguese name unique      : True
English name unique         : True
English names with '_'      : 56

names identical in both languages: 7


,product_category_name,product_category_name_english
19,consoles_games,consoles_games
20,audio,audio
22,cool_stuff,cool_stuff
29,pet_shop,pet_shop
31,market_place,market_place
60,la_cuisine,la_cuisine
65,dvds_blu_ray,dvds_blu_ray


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


- 71 rows and 2 columns, no missing values, no duplicates. Both the Portuguese and the English name are unique, so the table is a clean one-to-one lookup.
- 7 names are identical in both languages (`consoles_games`, `audio`, `cool_stuff`, `pet_shop`, `market_place`, `la_cuisine`, `dvds_blu_ray`). They are already English words used in Portuguese, so there is no translation error.
- 56 English names use underscores, like the Portuguese ones.
- The products table has 73 distinct categories and this table has 71 rows. So at least 2 product categories probably have no English name. Which ones is checked after the joins.

#### Summary
**The story in short.** The products table is a clean key table with one row per product, but 610 products (1.85%) have no listing information at all, and 2 have no weight or size. Weight and size depend strongly on the category, and both are right-skewed with real heavy products. Categories are concentrated (top 10 hold 64.1%) with a long thin tail, and the translation table is a clean lookup that is probably 2 categories short.

**Data quality issues**

| # | Issue | Records | What it is | Action | Cost of the alternative |
|---|---|---|---|---|---|
| 1 | Listing block missing (category, name length, description length, photos) | 610 (1.85%) | Collection gap, all four fields together | Keep, set `no_listing_info`, leave the category as unknown | Imputing invents a category and text sizes |
| 2 | Weight and dimensions missing | 2 | Collection gap | Keep, leave empty | Imputing from the overall median ignores the category effect |
| 3 | Weight equal to 0 | 4 (all `cama_mesa_banho`) | Impossible value | Keep, treat as unknown in the pipeline | Replacing with a number invents a weight |
| 4 | Suspicious heavy item | 1 (40,425 g, small box) | Possible entry error | Keep, cap in the pipeline | Dropping removes a real order's product |
| 5 | Misspelled column names | 2 columns | Source typo | Rename in the working copy | Mixed spelling causes lookup errors |
| 6 | Products identical in all attributes | 695 extra copies | Same values, different id | Keep | Merging is a guess |
| 7 | Long thin category tail | 13 categories with 20 or fewer products | Rare categories | Group rare categories at model time | Too few rows to learn from |
| 8 | Translation has 71 names for 73 categories | 2 categories (to be named after the joins) | Missing English names | Check after the joins | English labels would show as empty for them |

**Checks planned after the joins** (not done here on purpose)
- Which 2 product categories have no English name in the translation table.
- How many item lines and orders belong to the 610 products without listing information, and whether their bad-review rate differs.
- Weight and box size against freight value (is freight consistent with the declared size?).
- Category against delay and bad-review rates (for example furniture).

**What can be used at order approval (for Part 2)**

| Field | Known at approval? | Note |
|---|---|---|
| `product_category_name`, `no_listing_info` | Yes | Set at listing time. The category is the main product feature. |
| `product_weight_g`, dimensions, `volume_cm3` | Yes | Declared by the seller at listing time. They may be wrong (see MOP-8 on declared vs actual weight). |
| `product_photos_qty`, name and description length | Yes | Part of the listing |
| Category English name | Yes | Lookup only, for readable output |